In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
orders = pd.read_csv("data/raw/olist_orders_dataset.csv")

items = pd.read_csv("data/raw/olist_order_items_dataset.csv")

products = pd.read_csv("data/raw/olist_products_dataset.csv")

customers = pd.read_csv("data/raw/olist_customers_dataset.csv")

payments = pd.read_csv("data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("data/raw/olist_order_reviews_dataset.csv")

sellers = pd.read_csv("data/raw/olist_sellers_dataset.csv")

geo = pd.read_csv("data/raw/olist_geolocation_dataset.csv")

category = pd.read_csv("data/raw/product_category_name_translation.csv")

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [4]:
def missing_values(df, name):
    print("="*50)
    print(name)
    print("="*50)
    print(df.isnull().sum())

In [5]:
missing_values(orders, "Orders")
missing_values(items, "Order Items")
missing_values(products, "Products")
missing_values(customers, "Customers")
missing_values(payments, "Payments")
missing_values(reviews, "Reviews")
missing_values(sellers, "Sellers")
missing_values(geo, "Geolocation")
missing_values(category, "Category")

Orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Order Items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Products
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state    

In [6]:
datasets = {
    "Orders": orders,
    "Order Items": items,
    "Products": products,
    "Customers": customers,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Geo": geo,
    "Category": category
}

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates")

Orders: 0 duplicates
Order Items: 0 duplicates
Products: 0 duplicates
Customers: 0 duplicates
Payments: 0 duplicates
Reviews: 0 duplicates
Sellers: 0 duplicates
Geo: 261831 duplicates
Category: 0 duplicates


In [7]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [9]:
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_cols:
    products[col] = products[col].fillna(products[col].median())

In [10]:
products["product_category_name"] = products["product_category_name"].fillna("Unknown")

In [11]:
products = products.merge(
    category,
    on="product_category_name",
    how="left"
)

In [12]:
products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna("Unknown")
)

In [13]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [14]:
geo_clean = (
    geo.groupby("geolocation_zip_code_prefix")
       .agg({
           "geolocation_lat": "mean",
           "geolocation_lng": "mean",
           "geolocation_city": "first",
           "geolocation_state": "first"
       })
       .reset_index()
)

In [15]:
items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [16]:
items[items["price"] <= 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [17]:
items = items[items["price"] > 0]

In [18]:
items[items["freight_value"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [19]:
orders["order_id"].is_unique

True

In [20]:
customers["customer_id"].is_unique

True

In [21]:
products["product_id"].is_unique

True

In [27]:
orders.to_csv("data/cleaned/orders_clean.csv", index=False)

items.to_csv("data/cleaned/order_items_clean.csv", index=False)

products.to_csv("data/cleaned/products_clean.csv", index=False)

customers.to_csv("data/cleaned/customers_clean.csv", index=False)

payments.to_csv("data/cleaned/payments_clean.csv", index=False)

reviews.to_csv("data/cleaned/reviews_clean.csv", index=False)

sellers.to_csv("data/cleaned/sellers_clean.csv", index=False)

geo_clean.to_csv("data/cleaned/geolocation_clean.csv", index=False)